# Wealth Distribution From Fair Rock-Paper-Scissors Games

This notebook simulates 1,000 people who each begin with $100. At each round, two people are randomly selected to play a fair game. The loser pays the winner $1, unless the loser has already reached $0, in which case no transfer occurs. The notebook saves one half-resolution histogram frame every 10,000 games, for a total of 1,000,000 games.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image

FRAME_DIR = ROOT / "images" / "wealth_rps_frames"
CLEAR_OLD_FRAMES = False

FRAME_DIR.mkdir(parents=True, exist_ok=True)
if CLEAR_OLD_FRAMES:
    for old_frame in FRAME_DIR.glob("wealth_rps_*.png"):
        old_frame.unlink()

N_PEOPLE = 1000
INITIAL_WEALTH = 100
BET = 1
N_GAMES = 10_000_000
FRAME_EVERY = 10_000
SEED = 42

sns.set_theme(
    context="talk",
    style="whitegrid",
    rc={
        "axes.facecolor": "#f7f3ed",
        "figure.facecolor": "#f7f3ed",
        "grid.color": "#d7cfc4",
        "axes.edgecolor": "#2f3842",
        "axes.labelcolor": "#202832",
        "xtick.color": "#202832",
        "ytick.color": "#202832",
        "font.family": "DejaVu Sans",
    },
)

In [2]:
def gini(values):
    values = np.sort(np.asarray(values))
    n = len(values)
    return (2 * np.sum(np.arange(1, n + 1) * values)) / (n * values.sum()) - (n + 1) / n


def top_share(values, n_top):
    values = np.asarray(values)
    return np.sort(values)[-n_top:].sum() / values.sum()


def simulate():
    rng = np.random.default_rng(SEED)
    wealth = np.full(N_PEOPLE, INITIAL_WEALTH, dtype=np.int64)
    snapshots = [(0, wealth.copy())]

    for game in range(1, N_GAMES + 1):
        i, j = rng.choice(N_PEOPLE, size=2, replace=False)
        winner, loser = (i, j) if rng.random() < 0.5 else (j, i)

        if wealth[loser] > 0:
            wealth[loser] -= BET
            wealth[winner] += BET

        if game % FRAME_EVERY == 0:
            snapshots.append((game, wealth.copy()))

    return snapshots


snapshots = simulate()
len(snapshots), snapshots[-1][0], snapshots[-1][1].sum()

(1001, 10000000, np.int64(100000))

In [6]:
bins = np.arange(0, 701, 30)
x = np.linspace(0, bins[-1], 500)
bin_width = bins[1] - bins[0]
mean_wealth = INITIAL_WEALTH
boltzmann_counts = N_PEOPLE * bin_width * (1 / mean_wealth) * np.exp(-x / mean_wealth)


def draw_frame(game, wealth):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=90)

    sns.histplot(
        wealth,
        bins=bins,
        stat="count",
        color="#0f766e",
        edgecolor="#f7f3ed",
        linewidth=1.1,
        alpha=0.92,
        ax=ax,
    )
    ax.plot(x, boltzmann_counts, color="#b91c1c", linewidth=3.0, label="Boltzmann-like exponential")
    ax.axvline(mean_wealth, color="#1d4ed8", linewidth=2.2, linestyle="--", label="Average wealth")

    ax.set_title(f"Fair Random Exchange After {game:,} Games", loc="left", pad=16, fontweight="bold")
    ax.set_xlabel("Wealth ($)")
    ax.set_ylabel("Number of people")
    ax.set_xlim(0, bins[-1])
    #ax.set_ylim(0, 1000)
    ax.legend(frameon=False, loc="upper right", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)

    stats = (
        f"Gini: {gini(wealth):.2f}\n"
        f"Top 10 share: {top_share(wealth, 10):.0%}\n"
        f"At $0: {np.count_nonzero(wealth == 0)} people\n"
        f"Richest: ${wealth.max():,.0f}"
    )
    ax.text(
        0.985,
        0.60,
        stats,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=11,
        color="#202832",
        bbox={"boxstyle": "round,pad=0.45", "facecolor": "#fffaf2", "edgecolor": "#d7cfc4"},
    )

    fig.tight_layout()
    output = FRAME_DIR / f"wealth_rps_{game:07d}.png"
    fig.savefig(output)
    plt.close(fig)
    return output


frame_paths = [draw_frame(game, wealth) for game, wealth in snapshots]
len(frame_paths), frame_paths[0], frame_paths[-1]

(1001,
 PosixPath('/home/zhelunli/JSDA/images/wealth_rps_frames/wealth_rps_0000000.png'),
 PosixPath('/home/zhelunli/JSDA/images/wealth_rps_frames/wealth_rps_10000000.png'))

The frames are saved in `images/wealth_rps_frames/`. The GIF is saved as `images/wealth_rps_simulation.gif`.

In [5]:
gif_path = ROOT / "images" / "wealth_rps_simulation.gif"
frame_dir = ROOT / "images" / "wealth_rps_frames"
MAX_GIF_GAME = 7_000_000
frame_paths = [
    path for path in sorted(frame_dir.glob("wealth_rps_*.png"))
    if int(path.stem.split("_")[-1]) <= MAX_GIF_GAME
]
gif_frames = [Image.open(path).convert("P", palette=Image.Palette.ADAPTIVE) for path in frame_paths]
gif_frames[0].save(
    gif_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=20,
    loop=0,
    optimize=True,
)
gif_path, len(frame_paths), frame_paths[-1]

(PosixPath('/home/zhelunli/JSDA/images/wealth_rps_simulation.gif'),
 701,
 PosixPath('/home/zhelunli/JSDA/images/wealth_rps_frames/wealth_rps_7000000.png'))